# Proyecto Wearable - Fase 3: Preparación y Limpieza de Datos (CRISP-DM)

In [10]:
import pandas as pd
import sys
from pathlib import Path

# 1. Agregamos la raíz del proyecto al path de Python para que reconozca 'src'
raiz_proyecto = Path.cwd()
if not (raiz_proyecto / "src").is_dir():
    raiz_proyecto = raiz_proyecto.parent
if str(raiz_proyecto) not in sys.path:
    sys.path.insert(0, str(raiz_proyecto))

# 2. Recargamos el módulo para usar los cambios recientes aunque el kernel siga abierto
import importlib
import src.Limpieza.Sensores_limpios as sensores_limpios
importlib.reload(sensores_limpios)
SensoresDataCleaner = sensores_limpios.SensoresDataCleaner

# 3. Instanciamos con rutas basadas en la raíz del proyecto
limpiador = SensoresDataCleaner(raiz_proyecto / "Datacrude" / "01NoProcessData" / "SENSORES(sucio).xlsx", factorIqr=1.5)

# 4. Ejecutamos el pipeline completo
dfClean = limpiador.ejecutarPipeline()

# 5. Exportamos el resultado intermedio a tu carpeta correspondiente
limpiador.exportarCsv(raiz_proyecto / "Datacrude" / "02IntermediateData" / "Sensores_limpio.csv")

=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===
Dimensiones: (940, 17)
Nulos explícitos (isna): 1879
Duplicados exactos: 0

Columnas fantasma (artefactos de Excel): ['Unnamed: 15', 'Unnamed: 16']

Días de 'no uso' del tracker (0 pasos y 1440 min): 72
[PASO 1] Columnas fantasma eliminadas: ['Unnamed: 15', 'Unnamed: 16']
Nuevas dimensiones: (940, 15)
[PASO 2] 'ActivityDate' convertida a datetime.
Fechas inválidas (NaT): 0
[PASO SUEÑO] Duplicados de sueño eliminados: 3
[PASO INTEGRACIÓN] Columnas consolidadas: 18
[PASO 3] Registros de 'No Uso' eliminados: 72
Filas restantes: 868
[PASO 5] Aplicando regla IQR (winsorización):
  -> 'TotalSteps': IQR=6230.25 | límites=[0.00, 20400.38] | recortados=15
  -> 'Calories': IQR=976.75 | límites=[388.62, 4295.62] | recortados=11
  -> 'VeryActiveMinutes': IQR=35.00 | límites=[0.00, 87.50] | recortados=56
[PASO 6] 'Nivel_Pasos' -> valores únicos: ['Alta Actividad', 'Baja Actividad']
[PASO 6] 'Nivel_Sedentarismo' -> valores únicos: ['Alto Sedentarismo', 

## 1. Carga del Dataset Crudo desde Datacrude/01NoProcessData/

In [11]:
from pathlib import Path
import pandas as pd

def encontrar_excel():
    candidatos = [
        Path("../Datacrude/01NoProcessData/SENSORES(sucio).xlsx"),
        Path("Datacrude/01NoProcessData/SENSORES(sucio).xlsx"),
        Path.cwd() / "Datacrude" / "01NoProcessData" / "SENSORES(sucio).xlsx",
        Path.cwd().parent / "Datacrude" / "01NoProcessData" / "SENSORES(sucio).xlsx",
    ]
    for c in candidatos:
        if c.exists():
            return c
    raise FileNotFoundError("No se encontro SENSORES(sucio).xlsx")

ruta_excel = encontrar_excel()
dfSensores = pd.read_excel(ruta_excel)
print("Dimensiones iniciales:", dfSensores.shape)
dfSensores.head(3)

Dimensiones iniciales: (940, 17)


,Id,ActivityDate,TotalSteps,TotalDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Nivel_Pasos,Nivel_Sedentarismo,Unnamed: 15,Unnamed: 16
0,1503960366,2016-12-04 00:00:00,13162,8.500000e+00,1.880000e+14,0.55,6.060000e+14,0.0,25,13,328,728,1985,Alta Actividad,Alto Sedentarismo,NaN,NaN
1,1503960366,4/13/2016,10735,6.970000e+14,1.570000e+14,0.69,4.710000e+14,0.0,21,19,217,776,1797,Alta Actividad,Alto Sedentarismo,NaN,NaN
2,1503960366,4/14/2016,10460,6.740000e+14,2.440000e+14,0.40,3.910000e+14,0.0,30,11,181,1218,1776,Alta Actividad,Alto Sedentarismo,NaN,NaN


## 2. Instanciación y Ejecución del Pipeline de Limpieza

In [12]:
limpiador = SensoresDataCleaner(dfSensores, factorIqr=1.5)

# Ejecucion del pipeline paso a paso (diagnóstico, columnas fantasma, fechas, sueño, MNAR, IQR, categorías)
dfClean = limpiador.ejecutarPipeline()

=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===
Dimensiones: (940, 17)
Nulos explícitos (isna): 1879
Duplicados exactos: 0

Columnas fantasma (artefactos de Excel): ['Unnamed: 15', 'Unnamed: 16']

Días de 'no uso' del tracker (0 pasos y 1440 min): 72
[PASO 1] Columnas fantasma eliminadas: ['Unnamed: 15', 'Unnamed: 16']
Nuevas dimensiones: (940, 15)
[PASO 2] 'ActivityDate' convertida a datetime.
Fechas inválidas (NaT): 0
[PASO SUEÑO] Duplicados de sueño eliminados: 3
[PASO INTEGRACIÓN] Columnas consolidadas: 18
[PASO 3] Registros de 'No Uso' eliminados: 72
Filas restantes: 868
[PASO 5] Aplicando regla IQR (winsorización):
  -> 'TotalSteps': IQR=6230.25 | límites=[0.00, 20400.38] | recortados=15
  -> 'Calories': IQR=976.75 | límites=[388.62, 4295.62] | recortados=11
  -> 'VeryActiveMinutes': IQR=35.00 | límites=[0.00, 87.50] | recortados=56
[PASO 6] 'Nivel_Pasos' -> valores únicos: ['Alta Actividad', 'Baja Actividad']
[PASO 6] 'Nivel_Sedentarismo' -> valores únicos: ['Alto Sedentarismo', 

## 3. Exportación del Dataset Limpio a Datacrude/02IntermediateData/

In [13]:
limpiador.exportarCsv("Datacrude/02IntermediateData/Sensores_limpio.csv")

Dataset limpio exportado exitosamente a: C:\Users\Juan Sebastian\OneDrive\Documentos\MineriaDatosUP\Data-MinerProyecto\Datacrude\02IntermediateData\Sensores_limpio.csv (868 filas)


In [14]:
#bloque para confirmar que el código funciona correctamente
from src.Limpieza.Sensores_limpios import SensoresDataCleaner

# Instanciamos la clase con el dataset crudo
limpiador = SensoresDataCleaner("../Datacrude/01NoProcessData/SENSORES(sucio).xlsx", factorIqr=1.5)

# Ejecutamos el pipeline completo
dfClean = limpiador.ejecutarPipeline()

# Exportamos el resultado intermedio
limpiador.exportarCsv("../Datacrude/02IntermediateData/Sensores_limpio.csv")

=== PASO 0: DIAGNÓSTICO INICIAL DE CALIDAD ===
Dimensiones: (940, 17)
Nulos explícitos (isna): 1879
Duplicados exactos: 0

Columnas fantasma (artefactos de Excel): ['Unnamed: 15', 'Unnamed: 16']

Días de 'no uso' del tracker (0 pasos y 1440 min): 72
[PASO 1] Columnas fantasma eliminadas: ['Unnamed: 15', 'Unnamed: 16']
Nuevas dimensiones: (940, 15)
[PASO 2] 'ActivityDate' convertida a datetime.
Fechas inválidas (NaT): 0
[PASO SUEÑO] Duplicados de sueño eliminados: 3
[PASO INTEGRACIÓN] Columnas consolidadas: 18
[PASO 3] Registros de 'No Uso' eliminados: 72
Filas restantes: 868
[PASO 5] Aplicando regla IQR (winsorización):
  -> 'TotalSteps': IQR=6230.25 | límites=[0.00, 20400.38] | recortados=15
  -> 'Calories': IQR=976.75 | límites=[388.62, 4295.62] | recortados=11
  -> 'VeryActiveMinutes': IQR=35.00 | límites=[0.00, 87.50] | recortados=56
[PASO 6] 'Nivel_Pasos' -> valores únicos: ['Alta Actividad', 'Baja Actividad']
[PASO 6] 'Nivel_Sedentarismo' -> valores únicos: ['Alto Sedentarismo', 